# Bibliometria Analítica e Indicadores Científicos
### Atividade prática — Equipe 7 · `bibliometrix` & Biblioshiny (versão Google Colab)

---

### ⚙️ Antes de começar — ative o runtime em R (passo obrigatório)

Este notebook roda em **R**, não em Python. No menu do Colab:

> **Ambiente de execução → Alterar o tipo de ambiente de execução → Tipo de ambiente: `R` → Salvar**
> *(em inglês: Runtime → Change runtime type → Runtime type: R)*

Depois, rode a primeira célula para confirmar. Se aparecer a versão do R, está tudo certo.

> **ℹ️ Sobre o Biblioshiny:** a interface gráfica (`biblioshiny()`) **não funciona no Colab** — ela
> abre um servidor web local. No Colab usamos as **mesmas funções** que estão por trás de cada tela
> do Biblioshiny. Para usar a interface gráfica, instale o R/RStudio no seu computador.


In [ ]:
# Confirma que o runtime está em R
R.version.string
cat("Se você está vendo a versão do R acima, o runtime está correto.\n")

## 1 · Instalação dos pacotes

No Colab os pacotes são instalados **a cada nova sessão**. A linha de `options(repos=...)` usa
binários Linux prontos para acelerar bastante a instalação (de ~15 min para ~2–3 min).

> Se a instalação falhar, **comente** a linha `options(...)` (coloque `#` na frente) e rode de novo —
> ela voltará ao CRAN padrão (mais lento, porém robusto).


In [ ]:
# Acelera usando binários Linux (Colab = Ubuntu 22.04 'jammy'). Opcional.
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))

# Instala só o que faltar
pkgs <- c("bibliometrix", "bibliometrixData")
novos <- pkgs[!sapply(pkgs, requireNamespace, quietly = TRUE)]
if (length(novos)) install.packages(novos)

library(bibliometrix)
cat("Pacotes prontos. Versão do bibliometrix:",
    as.character(packageVersion("bibliometrix")), "\n")

## 2 · Conceito: Biblioshiny *vs* bibliometrix

| No Biblioshiny (clicando) | No bibliometrix (neste notebook) |
|---|---|
| Importar arquivo | `convert2df()` |
| Main Information | `biblioAnalysis()` + `summary()` |
| Annual Scientific Production | `plot()` |
| Sources → Bradford's Law | `bradford()` |
| Authors → Lotka's Law | `lotka()` |
| Authors → Author Impact (H/G/M) | `Hindex()` |
| Authors' Production over Time | `authorProdOverTime()` |
| Most Cited Documents | `citations()` |

**Ideia central:** o Biblioshiny é didático e rápido; o código é reprodutível e auditável.


## 3 · Carregar o corpus de dados

Usaremos um corpus de exemplo já incluído no pacote, para todos conseguirem executar sem download.
No trabalho final, você troca este passo pela importação dos seus arquivos (próxima seção).


In [ ]:
data(management, package = "bibliometrixData")
M <- management

dim(M)                          # documentos (linhas) x campos (colunas)
range(M$PY, na.rm = TRUE)       # intervalo de anos do corpus

### Como importar o SEU corpus no Colab (modelo para o trabalho final)

1. Clique no ícone de **pasta** 📁 na barra lateral esquerda do Colab.
2. Clique em **Fazer upload** e envie o arquivo exportado (ex.: `scopus.bib`).
3. O arquivo fica em `/content/`. Use esse caminho no `convert2df()`.

A célula abaixo está desativada (`if (FALSE)`) para não dar erro agora — ative quando tiver o arquivo.


In [ ]:
if (FALSE) {
  M <- convert2df(
    file     = "/content/scopus.bib",
    dbsource = "scopus",   # "wos", "scopus", "openalex", "dimensions", "lens", "pubmed"
    format   = "bibtex"    # "bibtex", "plaintext", "csv", "api"
  )
}

## 4 · Análise descritiva (a base de tudo)

`biblioAnalysis()` calcula as medidas; `summary()` as exibe e guarda em tabelas reutilizáveis.

> **🧪 Sua vez 1:** na tabela `MainInformationDF`, localize o **Annual Growth Rate** e a **média de
> citações por documento**. Anote os dois — são indicadores centrais do tema.


In [ ]:
results <- biblioAnalysis(M, sep = ";")
S <- summary(results, k = 10, pause = FALSE, verbose = FALSE)

S$MainInformationDF

## 5 · Indicador 1 — Produção científica anual (evolução temporal)

> **🧪 Sua vez 2:** identifique o ano de maior produção e descreva em uma frase o padrão de
> crescimento (linear, exponencial, com queda recente?).


In [ ]:
# Gráficos descritivos (produção anual, países, autores, etc.)
plot(results, k = 10, pause = FALSE)

In [ ]:
# Só a tabela ano a ano (útil para a tabela-síntese)
S$AnnualProduction

## 6 · Indicador 2 — Fontes/periódicos e Lei de Bradford

A **Lei de Bradford** identifica o núcleo (*Core*) de periódicos que concentra a maior parte dos
artigos — útil para focar leituras e submissões.

> **🧪 Sua vez 3:** quantos periódicos formam a zona **Core** do seu corpus? Liste os 3 principais.


In [ ]:
BR <- bradford(M)
head(BR$table, 10)

## 7 · Indicador 3 — Autores e Lei de Lotka

A **Lei de Lotka** descreve a concentração da produtividade: poucos autores publicam muito, muitos
publicam pouco. O `Beta` teórico é **2** — quanto mais próximo, mais o campo segue o padrão clássico.

> **🧪 Sua vez 4:** o `Beta` do seu corpus está próximo de 2? O que um desvio grande sugeriria?


In [ ]:
# A assinatura de lotka() mudou entre versões do bibliometrix:
#   versão atual (CRAN 5.x): lotka(M)        -> M = data frame, classe "bibliometrixDB"
#   versões antigas:         lotka(results)  -> saída de biblioAnalysis, classe "bibliometrix"
# O bloco abaixo escolhe automaticamente a forma que funciona:
invisible(capture.output(L <- lotka(M)))    # tenta a forma atual
if (!is.list(L)) L <- lotka(results)        # se vier NA, usa a forma antiga

L$AuthorProd
cat("\nBeta:", round(L$Beta, 3),
    "| R2:", round(L$R2, 3),
    "| p-valor (KS):", round(L$p.value, 4), "\n")

### Produção dos principais autores ao longo do tempo

> **🧪 Sua vez 5:** qual autor tem a trajetória mais longa? E qual concentra a produção em poucos anos?


In [ ]:
topAU <- authorProdOverTime(M, k = 10, graph = TRUE)
head(topAU$dfAU)

## 8 · Indicador 4 — Impacto: índices H, G e M

- **h-index:** *h* artigos com pelo menos *h* citações cada.
- **g-index:** dá mais peso aos artigos mais citados.
- **m-index:** h-index ÷ anos de carreira (corrige por tempo).

> **🧪 Sua vez 6:** o autor mais **produtivo** é também o de maior **impacto** (h-index)? O que isso
> diz sobre a diferença entre *quantidade* e *influência*?


In [ ]:
autores <- gsub(",", " ", names(results$Authors)[1:10])
H <- Hindex(M, field = "author", elements = autores, sep = ";", years = Inf)
H$H[order(-H$H$h_index), ]

## 9 · Indicador 5 — Documentos e referências mais citados

> **🧪 Sua vez 7:** as referências mais citadas costumam ser os "clássicos" da área. Identifique 2.


In [ ]:
CIT <- citations(M, field = "article", sep = ";")
head(CIT$Cited, 10)

## 10 · Tabela-síntese de indicadores (entregável principal)

Consolida os indicadores em uma só tabela — o coração do tema "indicadores científicos".


In [ ]:
agr <- S$MainInformationDF$V2[S$MainInformationDF$V1 == "Annual Growth Rate %"][1]

sintese <- data.frame(
  Indicador = c("Documentos (total)", "Autores", "Período coberto",
                "Crescimento anual (%)", "Média de citações por documento",
                "Documentos por autor"),
  Valor = c(results$Articles,
            length(results$Authors),
            paste(range(M$PY, na.rm = TRUE), collapse = " - "),
            agr,
            round(mean(as.numeric(M$TC), na.rm = TRUE), 2),
            round(results$Articles / length(results$Authors), 2))
)
sintese

# Exportar (aparece na pasta 📁 do Colab; baixe com o menu de 3 pontos do arquivo)
write.csv(sintese, "indicadores.csv", row.names = FALSE)
cat("Arquivo 'indicadores.csv' salvo em /content/\n")

## 11 · E o Biblioshiny?

No Colab rodamos as funções; para usar a **interface gráfica**, no seu computador:

```r
install.packages("bibliometrix")
library(bibliometrix)
biblioshiny()      # abre no navegador
```

**Mapa de menus → função** (para conferir os números que você calculou aqui):

| Menu no Biblioshiny | Função |
|---|---|
| Data → Load Data | `convert2df` |
| Overview → Main Information | `biblioAnalysis` + `summary` |
| Sources → Bradford's Law | `bradford` |
| Authors → Lotka's Law | `lotka` |
| Authors → Author Impact | `Hindex` |
| Documents → Most Cited | `citations` |


## 12 · Checklist de entrega

Seguindo o **Padrão de Entrega** da capacitação:

- [ ] **Versão editável** — este `.ipynb` (o Colab salva no seu Google Drive).
- [ ] **PDF obrigatório** — `Arquivo → Imprimir → Salvar como PDF`.
- [ ] Tabela-síntese exportada (`indicadores.csv`).
- [ ] 1 gráfico temporal + 1 de autores.
- [ ] 1 print do Biblioshiny conferindo um indicador.
- [ ] Cheat sheet (1–2 páginas) anexado.

---

### Desafios finais (discussão da equipe)
1. O campo segue a Lei de Lotka (Beta ≈ 2)?
2. O autor mais produtivo é o mais influente?
3. A produção anual indica ascensão, maturidade ou declínio?
4. Qual indicador você defenderia como o **mais importante** para avaliar um corpus?

*Aria, M. & Cuccurullo, C. (2017). bibliometrix. Journal of Informetrics, 11(4), 959-975.*
